In [119]:
import cv2
import matplotlib.pyplot as plt
import numpy as np

In [120]:

numTeste = [12, 21, 30, 32]
numTreino = [1, 2, 4, 9, 10, 11, 13, 22, 23, 25]

testes = [cv2.imread(f"treino/lego{n}.jpg", 0) for n in numTeste]
treino = [cv2.imread(f"treino/lego0{n}.jpg" if n < 10 else f"treino/lego{n}.jpg", 0) for n in numTreino]


Em tons de cinzento, o fundo azul e algumas peças (vermelha, verde, laranja...) ficam com brilho parecido, e nenhum limiar, fixo ou Otsu, as consegue separar. O problema não é o valor do limiar, é que o brilho não distingue fundo de peças. A solução é segmentar pela cor. Ou seja, em ves de ler a imagem em tons de cinzento, esta sera lida com as cores originais e usa-se o fundo azul como referância. 

Porque funciona: no fundo azul, B é bastante maior que R e G, mesmo nas zonas de sombra (fica mais escuro, mas continua azul). Nas peças isso não acontece: vermelha e laranja têm R alto, a verde tem G alto, a amarela tem R e G altos, e a bege/branca/cinzenta tem os três canais parecidos, logo B não ganha por margem. Os valores 40 e 20 são o ponto de partida, e deves ajustá-los vendo as máscaras.

O 255 - fundo funciona porque fundo é um array uint8 só com 0 e 255: onde era 255 (fundo) passa a 0, e onde era 0 (peça) passa a 255. Assim, o treino_bin tem todas as imagens em binario, ou seja, peças a branco e fundo a preto.

In [121]:

#Binarização

#treino_bin = [cv2.threshold(img, 140, 255, cv2.THRESH_BINARY)[1] for img in treino]

def segmentar(img):
    fundo = cv2.inRange(img, (100, 0, 0), (255, 140, 100))   # (B, G, R) min e max
    return 255 - fundo                                        # peças = branco, fundo = preto


treino_bin = [segmentar(img) for img in treino]


plt.imshow(treino_bin[0], cmap="gray")
plt.show()


hist = cv2.calcHist([treino[2]], [0], None, [256], [0, 256])

plt.bar(range(256), hist.ravel(), width=1)
plt.title("Histograma da imagem")
plt.xlabel("Nível de cinzento (0-255)")
plt.ylabel("Nº de pixels")
plt.show()

error: OpenCV(4.13.0) D:\a\opencv-python\opencv-python\opencv\modules\core\src\arithm.cpp:2064: error: (-209:Sizes of input arguments do not match) The lower boundary is neither an array of the same size and same type as src, nor a scalar in function 'cv::inRange'


In [ ]:
#Operador Morfológicos

